# Data Exploration

Norman et al. Perturb-seq dataset (K562 cells, CRISPRa), harmonized via [scPerturb](https://www.sanderlab.org/scPerturb/). Source: GSE133344, downloaded from [Zenodo record 13350497](https://zenodo.org/records/13350497).

In [1]:
import sys
sys.path.insert(0, '..')
import scanpy as sc
import pandas as pd

adata = sc.read_h5ad('../data/norman.h5ad')
adata

AnnData object with n_obs × n_vars = 111445 × 33694
    obs: 'guide_id', 'read_count', 'UMI_count', 'coverage', 'gemgroup', 'good_coverage', 'number_of_cells', 'tissue_type', 'cell_line', 'cancer', 'disease', 'perturbation_type', 'celltype', 'organism', 'perturbation', 'nperts', 'ngenes', 'ncounts', 'percent_mito', 'percent_ribo'
    var: 'ensemble_id', 'ncounts', 'ncells'

## Schema

`obs['perturbation']` is `'control'`, a single gene symbol, or `'GENE1_GENE2'` for two-gene combinations.

In [2]:
adata.obs[['perturbation', 'nperts', 'ngenes', 'ncounts', 'percent_mito']].head(10)

,perturbation,nperts,ngenes,ncounts,percent_mito
TTGAACGAGACTCGGA,ARID1A,1,3079,15097.0,5.815725
CGTTGGGGTGTTTGTG,BCORL1,1,2100,8551.0,4.104783
GAACCTAAGTGTTAGA,FOSB,1,2772,10999.0,5.655060
CCTTCCCTCCGTCATC,SET_KLF1,2,5385,38454.0,4.335050
TCAATCTGTCTTTCAT,OSR2,1,4869,27926.0,5.084867
TCCCGATGTCTCTTAT,KLF1_BAK1,2,3876,21433.0,6.531983
AAACCTGTCCAGAAGG,FOXA3_FOXL2,2,3454,16307.0,5.341264
CTGCCTAGTTCCACAA,TP73,1,2195,11678.0,5.206371
GAACCTATCCAGAAGG,HES7,1,2623,10527.0,10.990786
AAGCCGCTCACTCCTG,IRF1_SET,2,2979,11213.0,7.304022


In [3]:
perts = adata.obs['perturbation'].astype(str)
non_control = perts[perts != 'control']
singles = [p for p in non_control.unique() if '_' not in p]
combos = [p for p in non_control.unique() if '_' in p]
print(f'Total cells: {adata.n_obs}')
print(f'Unique perturbations (excl. control): {len(non_control.unique())}')
print(f'  single-gene: {len(singles)}')
print(f'  two-gene combos: {len(combos)}')
print(f'Control cells: {(perts == "control").sum()}')

Total cells: 111445
Unique perturbations (excl. control): 236
  single-gene: 105
  two-gene combos: 131
Control cells: 11855


Every gene used in a combo also appears as a single-gene perturbation in this dataset — useful later for constructing genuinely unseen-gene test splits (see `src/data/splits.py`).

In [4]:
genes_in_combos = set(g for c in combos for g in c.split('_'))
print(f'Genes appearing in combos: {len(genes_in_combos)}')
print(f'...of which also seen as singles: {len(genes_in_combos & set(singles))}')

Genes appearing in combos: 73
...of which also seen as singles: 73


In [5]:
perts.value_counts().head(15).to_frame('n_cells')

,n_cells
perturbation,
control,11855
KLF1,1960
BAK1,1457
CEBPE,1233
CEBPE_RUNX1T1,1219
UBASH3B,1202
ETS2,1201
TBX3_TBX2,1167
OSR2,1003
